# Expresiones Regulares
1. Leer el corpus
2. Expresiones Regulares

In [1]:

import os

path_1k = "G:/Mi unidad/ir26a-danielFlores/data/1knormalized"
documentos_gutenberg = []

for archivo in os.listdir(path_1k):
    ruta_completa = os.path.join(path_1k, archivo)
    with open(ruta_completa, "r", encoding="utf-8") as f:
        contenido = f.read()
        documentos_gutenberg.append(contenido)

print(f"Total: {len(documentos_gutenberg)} libros")


Total: 1000 libros


4. TF-IDF

In [2]:
def df_process(archivo):
    re.sub(r'[^\w\s]', '', archivo)
    tokens = re.findall(r'[a-zA-ZáéíóúÁÉÍÓÚñÑ]+', archivo)
    stemmer = SnowballStemmer('english')
    raices = [stemmer.stem(palabra) for palabra in tokens] # el resultado es un arreglo
    return ' '.join(raices)


# Resultado: ['corr', 'corr', 'corr', 'cas', 'casit', 'dorm']

#### Creación df_corpus

In [3]:
import pandas as pd
df_corpus = pd.DataFrame(documentos_gutenberg, columns=['contenido'])

#### Matriz TF-IDF

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF directo sobre el texto original
# stop_words='english' ya elimina palabras vacías
# token_pattern solo extrae palabras en minúsculas
vectorizador = TfidfVectorizer(
    stop_words='english',
    token_pattern=r'[a-z]+',       # sin stemming, igual de efectivo para búsqueda
    max_features=50000,             # limita vocabulario → más rápido
    min_df=2                        # ignora términos que aparecen en menos de 2 docs
)

tfidf_matrix = vectorizador.fit_transform(df_corpus['contenido'])
print(f"Matriz TF-IDF: {tfidf_matrix.shape}")


Matriz TF-IDF: (1000, 50000)


In [9]:
print(f"Matriz TF-IDF creada con forma: {tfidf_matrix.shape}")


Matriz TF-IDF creada con forma: (1000, 50000)


##### Consulta

In [11]:
query = "journey"
query_tfidf = vectorizador.transform([query]) 

##### Comparación con coseno

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

# dist es un numpy.ndarray de forma (n_docs,), independiente del DataFrame
dist = cosine_similarity(tfidf_matrix, query_tfidf).flatten()


In [13]:
import numpy as np

# Ordenar índices de mayor a menor similitud
indices_ordenados = np.argsort(dist)[::-1]

# Mostrar top-5 usando el índice para acceder a dist y al DataFrame
top_n = 5
print(f'Top {top_n} documentos más relevantes para: "{query}"\n')
for rank, idx in enumerate(indices_ordenados[:top_n], start=1):
    similitud = dist[idx]                                  # acceso por índice al array
    resumen   = df_corpus.iloc[idx]['contenido'][:80].replace('\n', ' ')
    print(f'{rank}. [idx={idx}]  similitud={similitud:.6f}  |  {resumen}...')


Top 5 documentos más relevantes para: "journey"

1. [idx=346]  similitud=0.073019  |  project gutenbergs a journey to the centre of the earth by jules verne    this e...
2. [idx=252]  similitud=0.041640  |   start of the project gutenberg ebook 42324                                  fra...
3. [idx=612]  similitud=0.037108  |  ﻿the project gutenberg ebook of the confessions of jean jacques rousseau by  jea...
4. [idx=18]  similitud=0.036708  |   start of the project gutenberg ebook 5197      my life  by richard wagner  in t...
5. [idx=144]  similitud=0.035647  |   start of the project gutenberg ebook 41445     transcribers note this text was ...


hacer que dist sea otra ed, un array. debo usar los indices para acceder a la información y no hacerlo en la misma matriz como otra columna